# WESAD Preprocessing

For each subject: load raw signals, slide a 60-second window across time, extract features from each valid window, then discard the raw signals before moving to the next subject.

Output: `X` (features), `y` (binary stress label), `groups` (subject ID for leave-one-subject-out CV).

Attribution: Claude used to generate feature extraction helpers and sanity checks. Manual editing was necessary to fix inefficiencies in feature extraction code and achieve ~10x speedup.

In [16]:
import gc
import pickle
from pathlib import Path

import numpy as np
import neurokit2 as nk
from scipy import signal as sp_signal
from scipy.stats import linregress as sp_linregress

DATA_DIR    = Path("../data/WESAD")
SUBJECT_IDS = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]

FS_CHEST  = 700
FS_BVP    = 64
FS_EDA_W  = 4
FS_TEMP_W = 4
FS_ACC_W  = 32

LABEL_NAMES = {0: "transient", 1: "baseline", 2: "stress", 3: "amusement", 4: "meditation"}

WINDOW_SEC = 60.0
SHIFT_SEC  = 0.25
WIN_C      = int(WINDOW_SEC * FS_CHEST)   # 42 000 chest samples
STEP_C     = int(SHIFT_SEC  * FS_CHEST)   # 175   chest samples

KEEP_LABELS = {1, 2, 3}   # baseline, stress, amusement

print(f"Window : {WIN_C} samples  ({WINDOW_SEC}s @ {FS_CHEST} Hz)")
print(f"Step   : {STEP_C} samples  ({SHIFT_SEC}s)")

Window : 42000 samples  (60.0s @ 700 Hz)
Step   : 175 samples  (0.25s)


### Feature extraction helpers

Each signal gets its own function returning a `dict[str, float]`. All functions wrap `neurokit2` / `scipy` calls in `try/except` and fall back to `np.nan` — the EBM handles missing values natively.

**Features computed:**
- **ECG → HRV** (time domain: mean/std HR, mean RR, SDNN, RMSSD, NN50, pNN50, TINN; frequency domain: LF, HF, LF/HF ratio, LF_norm, HF_norm, total power)
- **BVP → HRV** (same set from wrist PPG peaks)
- **Chest & wrist EDA** (raw stats + SCL tonic decomposition + SCR phasic peaks via neurokit2)
- **Respiration** (breath rate, inhale/exhale durations, IE ratio, stretch, inspiration volume)
- **Chest & wrist TEMP** (mean, std, min, max, range, slope)
- **EMG** (statistical, Welch frequency bands, peak envelope)
- **Chest & wrist ACC** (per-axis + magnitude stats + peak frequency)
- **Cross-signal** (RMSSD/HR normalised, HF/breath-rate coherence proxy)

In [ ]:
# ── Shared utilities ──────────────────────────────────────────────────────

def _slope(arr):
    x = np.arange(len(arr), dtype=np.float64)
    try:
        return float(sp_linregress(x, arr.astype(np.float64)).slope)
    except Exception:
        return np.nan


def _welch(arr, fs):
    """Return (mean_freq, median_freq, peak_freq, band_power_fn)."""
    try:
        nperseg = min(len(arr), max(256, len(arr) // 8))
        freqs, psd = sp_signal.welch(arr.astype(np.float64), fs=fs, nperseg=nperseg)
        total = psd.sum()
        if total < 1e-30:
            return np.nan, np.nan, np.nan, lambda lo, hi: np.nan
        mf   = float(np.sum(freqs * psd) / total)
        medf = float(freqs[np.searchsorted(np.cumsum(psd), total / 2)])
        pkf  = float(freqs[np.argmax(psd)])
        def bp(lo, hi, _f=freqs, _p=psd):
            m = (_f >= lo) & (_f < hi)
            return float(_p[m].sum()) if m.any() else 0.0
        return mf, medf, pkf, bp
    except Exception:
        return np.nan, np.nan, np.nan, lambda lo, hi: np.nan


# ── HRV helpers ───────────────────────────────────────────────────────────
# Manual edit of Claude code necessary: very inefficient approach from calling nk.hrv_frequecny for very window.
# nk.hrv_frequency is avoided as it costs ~27 ms/call due to DataFrame overhead.
# Manual implementation: interpolate RR -> Welch PSD -> band integrate -> 1 ms/call.

def _hrv_freq_manual(peaks, fs, prefix):
    """LF/HF HRV via Welch on linearly-interpolated RR series."""
    out = {f'{prefix}_{k}': np.nan
           for k in ('LF', 'HF', 'LFHF', 'LFn', 'HFn', 'TP')}
    if len(peaks) < 4:
        return out
    try:
        rr_ms    = np.diff(np.asarray(peaks, dtype=np.float64)) / fs * 1000.0
        t_beats  = np.cumsum(rr_ms) / 1000.0   # seconds
        interp_fs = 4.0
        t_interp  = np.arange(0.0, t_beats[-1], 1.0 / interp_fs)
        if len(t_interp) < 16:
            return out
        rr_i = np.interp(t_interp, t_beats, rr_ms)
        rr_i -= rr_i.mean()   # detrend
        nperseg = min(len(rr_i), max(32, len(rr_i) // 4))
        freqs, psd = sp_signal.welch(rr_i, fs=interp_fs, nperseg=nperseg)
        lf = float(psd[(freqs >= 0.04) & (freqs < 0.15)].sum())
        hf = float(psd[(freqs >= 0.15) & (freqs < 0.40)].sum())
        tp = float(psd.sum())
        out.update({
            f'{prefix}_LF':   lf,
            f'{prefix}_HF':   hf,
            f'{prefix}_LFHF': lf / (hf + 1e-10),
            f'{prefix}_TP':   tp,
            f'{prefix}_LFn':  lf / (tp + 1e-10),
            f'{prefix}_HFn':  hf / (tp + 1e-10),
        })
    except Exception:
        pass
    return out


def _hrv_from_peaks(peaks, fs, prefix):
    """Time-domain + frequency-domain HRV from an array of peak sample indices."""
    keys = [
        f'{prefix}_mean_HR', f'{prefix}_std_HR', f'{prefix}_mean_RR',
        f'{prefix}_SDNN',    f'{prefix}_RMSSD',  f'{prefix}_NN50',
        f'{prefix}_pNN50',
        f'{prefix}_LF',      f'{prefix}_HF',     f'{prefix}_LFHF',
        f'{prefix}_LFn',     f'{prefix}_HFn',    f'{prefix}_TP',
    ]
    out = dict.fromkeys(keys, np.nan)
    if len(peaks) < 4:
        return out
    try:
        rr_ms   = np.diff(np.asarray(peaks, dtype=np.float64)) / fs * 1000.0
        hr      = 60000.0 / rr_ms
        diff_rr = np.diff(rr_ms)
        nn50    = int(np.sum(np.abs(diff_rr) > 50))
        out.update({
            f'{prefix}_mean_HR': float(hr.mean()),
            f'{prefix}_std_HR':  float(hr.std()),
            f'{prefix}_mean_RR': float(rr_ms.mean()),
            f'{prefix}_SDNN':    float(rr_ms.std(ddof=1)),
            f'{prefix}_RMSSD':   float(np.sqrt(np.mean(diff_rr ** 2))),
            f'{prefix}_NN50':    float(nn50),
            f'{prefix}_pNN50':   float(nn50 / len(rr_ms) * 100),
        })
    except Exception:
        pass
    out.update(_hrv_freq_manual(peaks, fs, prefix))
    return out


# ── Subject-level pre-computation ─────────────────────────────────────────
# Each expensive nk call runs ONCE per subject, not once per window.
# Per-window code then just slices the resulting arrays.

_ECG_DS_FS  = 250     # Hz — R-peaks don't need 700 Hz
_RESP_FS_DS = 25      # Hz — breathing is 0.1–0.5 Hz; 25 Hz is plenty
_EDA_FS     = 4       # Hz — same rate used for wrist EDA

def precompute_subject(chest, wrist):
    pc = {}

    # ── ECG peaks at 250 Hz ───────────────────────────────────────────
    try:
        ecg_ds = sp_signal.resample_poly(
            chest['ECG'].astype(np.float64), _ECG_DS_FS, FS_CHEST)
        _, info = nk.ecg_peaks(ecg_ds, sampling_rate=_ECG_DS_FS)
        pc['ecg_peaks'] = np.asarray(info['ECG_R_Peaks'])
    except Exception:
        pc['ecg_peaks'] = np.array([], dtype=int)

    # ── BVP / PPG peaks at 64 Hz ──────────────────────────────────────
    try:
        _, info = nk.ppg_peaks(wrist['BVP'].astype(np.float64), sampling_rate=FS_BVP)
        pc['bvp_peaks'] = np.asarray(info['PPG_Peaks'])
    except Exception:
        pc['bvp_peaks'] = np.array([], dtype=int)

    # ── Chest EDA decomposed at 4 Hz ─────────────────────────────────
    try:
        eda_c = sp_signal.resample_poly(
            chest['EDA'].astype(np.float64), _EDA_FS, FS_CHEST)
        eda_c_df, _ = nk.eda_process(eda_c, sampling_rate=_EDA_FS)
        pc['eda_c_raw']    = eda_c
        pc['eda_c_tonic']  = eda_c_df['EDA_Tonic'].values
        pc['eda_c_phasic'] = eda_c_df['EDA_Phasic'].values
        pc['eda_c_df']     = eda_c_df
    except Exception:
        pc['eda_c_raw'] = pc['eda_c_tonic'] = pc['eda_c_phasic'] = pc['eda_c_df'] = None

    # ── Wrist EDA decomposed at 4 Hz ─────────────────────────────────
    try:
        eda_w_df, _ = nk.eda_process(wrist['EDA'].astype(np.float64), sampling_rate=FS_EDA_W)
        pc['eda_w_raw']    = wrist['EDA'].astype(np.float64)
        pc['eda_w_tonic']  = eda_w_df['EDA_Tonic'].values
        pc['eda_w_phasic'] = eda_w_df['EDA_Phasic'].values
        pc['eda_w_df']     = eda_w_df
    except Exception:
        pc['eda_w_raw'] = pc['eda_w_tonic'] = pc['eda_w_phasic'] = pc['eda_w_df'] = None

    # ── Respiration processed at 25 Hz ───────────────────────────────
    try:
        resp_ds = sp_signal.resample_poly(
            chest['Resp'].astype(np.float64), _RESP_FS_DS, FS_CHEST)
        rsp_df, rsp_info = nk.rsp_process(resp_ds, sampling_rate=_RESP_FS_DS)
        pc['resp_ds']      = resp_ds
        pc['rsp_rate']     = rsp_df['RSP_Rate'].values
        pc['rsp_phase']    = rsp_df['RSP_Phase'].values
        pc['rsp_peaks']    = np.asarray(rsp_info.get('RSP_Peaks',   []))
        pc['rsp_troughs']  = np.asarray(rsp_info.get('RSP_Troughs', []))
    except Exception:
        pc['resp_ds'] = pc['rsp_rate'] = pc['rsp_phase'] = None
        pc['rsp_peaks'] = pc['rsp_troughs'] = np.array([], dtype=int)

    return pc


# ── Per-window feature helpers (slice from precomputed data) ──────────────

def _eda_features_precomp(raw_win, tonic_win, phasic_win, df_win, prefix):
    """EDA features from pre-decomposed slices — no nk call needed."""
    keys = [
        f'{prefix}_EDA_mean',  f'{prefix}_EDA_std',   f'{prefix}_EDA_min',
        f'{prefix}_EDA_max',   f'{prefix}_EDA_range',  f'{prefix}_EDA_slope',
        f'{prefix}_SCL_mean',  f'{prefix}_SCL_std',
        f'{prefix}_SCL_slope', f'{prefix}_SCL_corr_time',
        f'{prefix}_n_SCR',     f'{prefix}_SCR_mean_amp', f'{prefix}_SCR_sum_amp',
        f'{prefix}_SCR_mean_dur', f'{prefix}_SCR_sum_dur', f'{prefix}_SCR_area',
    ]
    out = dict.fromkeys(keys, np.nan)
    try:
        arr = raw_win.astype(np.float64)
        out.update({
            f'{prefix}_EDA_mean':  float(arr.mean()),
            f'{prefix}_EDA_std':   float(arr.std()),
            f'{prefix}_EDA_min':   float(arr.min()),
            f'{prefix}_EDA_max':   float(arr.max()),
            f'{prefix}_EDA_range': float(arr.max() - arr.min()),
            f'{prefix}_EDA_slope': _slope(arr),
        })
    except Exception:
        pass
    try:
        tonic  = tonic_win.astype(np.float64)
        phasic = phasic_win.astype(np.float64)
        t      = np.arange(len(tonic), dtype=np.float64)
        out[f'{prefix}_SCL_mean']  = float(tonic.mean())
        out[f'{prefix}_SCL_std']   = float(tonic.std())
        out[f'{prefix}_SCL_slope'] = _slope(tonic)
        try:
            out[f'{prefix}_SCL_corr_time'] = float(np.corrcoef(t, tonic)[0, 1])
        except Exception:
            pass
        out[f'{prefix}_SCR_area'] = float(np.trapz(np.abs(phasic)))
    except Exception:
        pass
    try:
        if df_win is not None and 'SCR_Height' in df_win.columns:
            h     = df_win['SCR_Height'].values
            h_pos = h[h > 0]
            out[f'{prefix}_n_SCR'] = float(len(h_pos))
            if len(h_pos):
                out[f'{prefix}_SCR_mean_amp'] = float(h_pos.mean())
                out[f'{prefix}_SCR_sum_amp']  = float(h_pos.sum())
        if df_win is not None and 'SCR_Rise_Time' in df_win.columns:
            durs = df_win['SCR_Rise_Time'].values + df_win['SCR_Recovery_Time'].values
            durs = durs[durs > 0]
            if len(durs):
                out[f'{prefix}_SCR_mean_dur'] = float(durs.mean())
                out[f'{prefix}_SCR_sum_dur']  = float(durs.sum())
    except Exception:
        pass
    return out


_RESP_KEYS = [
    'resp_mean_inhale', 'resp_std_inhale', 'resp_mean_exhale', 'resp_std_exhale',
    'resp_IE_ratio', 'resp_rate', 'resp_rate_std', 'resp_stretch',
    'resp_inspiration_vol', 'resp_sum_dur',
]

def _resp_features_precomp(raw_win, resp_ds_win, rsp_rate_win, rsp_phase_win,
                            rsp_peaks_full, rsp_troughs_full, cs_r, ce_r):
    """Resp features from pre-processed arrays — slice peaks to window range."""
    out = dict.fromkeys(_RESP_KEYS, np.nan)
    try:
        out['resp_stretch'] = float(raw_win.astype(np.float64).max() -
                                    raw_win.astype(np.float64).min())
    except Exception:
        pass
    if rsp_rate_win is None:
        return out
    try:
        rate = rsp_rate_win[np.isfinite(rsp_rate_win)]
        if len(rate):
            out['resp_rate']     = float(rate.mean())
            out['resp_rate_std'] = float(rate.std())
        out['resp_inspiration_vol'] = float(
            np.trapz(np.abs(resp_ds_win[rsp_phase_win == 1])))
        # Slice peaks/troughs that fall inside this window
        pk  = rsp_peaks_full[(rsp_peaks_full  >= cs_r) & (rsp_peaks_full  < ce_r)]
        tr  = rsp_troughs_full[(rsp_troughs_full >= cs_r) & (rsp_troughs_full < ce_r)]
        if len(pk) > 1 and len(tr) > 1:
            inhale, exhale = [], []
            for p in pk:
                pre  = tr[tr < p];  
                if len(pre):  inhale.append((p - pre[-1])  / _RESP_FS_DS)
                post = tr[tr > p]
                if len(post): exhale.append((post[0] - p) / _RESP_FS_DS)
            if inhale:
                out['resp_mean_inhale'] = float(np.mean(inhale))
                out['resp_std_inhale']  = float(np.std(inhale))
            if exhale:
                out['resp_mean_exhale'] = float(np.mean(exhale))
                out['resp_std_exhale']  = float(np.std(exhale))
            if inhale and exhale:
                out['resp_IE_ratio'] = float(np.mean(inhale) / (np.mean(exhale) + 1e-8))
                out['resp_sum_dur']  = float(sum(inhale) + sum(exhale))
    except Exception:
        pass
    return out


# ── Simple per-signal helpers (unchanged — fast scipy ops) ────────────────

def _temp_features(temp_arr, prefix):
    a = temp_arr.astype(np.float64)
    return {
        f'{prefix}_TEMP_mean':  float(a.mean()),
        f'{prefix}_TEMP_std':   float(a.std()),
        f'{prefix}_TEMP_min':   float(a.min()),
        f'{prefix}_TEMP_max':   float(a.max()),
        f'{prefix}_TEMP_range': float(a.max() - a.min()),
        f'{prefix}_TEMP_slope': _slope(a),
    }


_EMG_BANDS = [(0,50),(50,100),(100,150),(150,200),(200,250),(250,300),(300,350)]
_EMG_KEYS  = (
    ['emg_mean','emg_std','emg_range','emg_median','emg_p10','emg_p90','emg_abs_integral',
     'emg_mean_freq','emg_median_freq','emg_peak_freq'] +
    [f'emg_psd_b{i+1}' for i in range(7)] +
    ['emg_n_peaks','emg_peak_mean_amp','emg_peak_std_amp']
)

def _emg_features(emg_arr):
    out = dict.fromkeys(_EMG_KEYS, np.nan)
    try:
        sos = sp_signal.butter(4, 20.0 / (FS_CHEST / 2), btype='high', output='sos')
        arr = sp_signal.sosfilt(sos, emg_arr.astype(np.float64))
    except Exception:
        arr = emg_arr.astype(np.float64)
    out.update({
        'emg_mean':         float(arr.mean()),
        'emg_std':          float(arr.std()),
        'emg_range':        float(arr.max() - arr.min()),
        'emg_median':       float(np.median(arr)),
        'emg_p10':          float(np.percentile(arr, 10)),
        'emg_p90':          float(np.percentile(arr, 90)),
        'emg_abs_integral': float(np.abs(arr).mean()),
    })
    mf, medf, pkf, bp = _welch(arr, FS_CHEST)
    out.update({'emg_mean_freq': mf, 'emg_median_freq': medf, 'emg_peak_freq': pkf})
    for i, (lo, hi) in enumerate(_EMG_BANDS):
        out[f'emg_psd_b{i+1}'] = bp(lo, hi)
    try:
        sos_lp = sp_signal.butter(4, 5.0 / (FS_CHEST / 2), btype='low', output='sos')
        env    = sp_signal.sosfilt(sos_lp, np.abs(arr))
        pidx, _ = sp_signal.find_peaks(env, height=float(env.mean()))
        out['emg_n_peaks'] = float(len(pidx))
        if len(pidx):
            amps = env[pidx]
            out['emg_peak_mean_amp'] = float(amps.mean())
            out['emg_peak_std_amp']  = float(amps.std())
    except Exception:
        pass
    return out


def _acc_features(acc_arr, fs, prefix):
    axes  = ('x', 'y', 'z')
    keys  = [f'{prefix}_ACC_{ax}_{s}' for ax in axes for s in ('mean','std','abs_int','peak_freq')]
    keys += [f'{prefix}_ACC_3D_{s}' for s in ('mean', 'std', 'abs_int')]
    out   = dict.fromkeys(keys, np.nan)
    try:
        mag = np.linalg.norm(acc_arr, axis=1)
        for i, ax in enumerate(axes):
            ch = acc_arr[:, i].astype(np.float64)
            _, _, pkf, _ = _welch(ch, fs)
            out.update({
                f'{prefix}_ACC_{ax}_mean':      float(ch.mean()),
                f'{prefix}_ACC_{ax}_std':       float(ch.std()),
                f'{prefix}_ACC_{ax}_abs_int':   float(np.abs(ch).mean()),
                f'{prefix}_ACC_{ax}_peak_freq': pkf,
            })
        out.update({
            f'{prefix}_ACC_3D_mean':    float(mag.mean()),
            f'{prefix}_ACC_3D_std':     float(mag.std()),
            f'{prefix}_ACC_3D_abs_int': float(np.abs(mag).mean()),
        })
    except Exception:
        pass
    return out


_CROSS_KEYS = ['cross_RMSSD_HR_norm', 'cross_HF_breath_rate']

def _cross_features(ecg_f, resp_f):
    out = dict.fromkeys(_CROSS_KEYS, np.nan)
    try:
        rmssd = ecg_f.get('ecg_RMSSD', np.nan)
        hr    = ecg_f.get('ecg_mean_HR', np.nan)
        if np.isfinite(rmssd) and np.isfinite(hr) and hr > 0:
            out['cross_RMSSD_HR_norm'] = float(rmssd / hr)
    except Exception:
        pass
    try:
        hf = ecg_f.get('ecg_HF', np.nan)
        br = resp_f.get('resp_rate', np.nan)
        if np.isfinite(hf) and np.isfinite(br) and br > 0:
            out['cross_HF_breath_rate'] = float(hf / br)
    except Exception:
        pass
    return out


# ── FEATURE_COLS ──────────────────────────────────────────────────────────

def _dummy_feature_cols():
    _nan_eda = {k: np.nan for k in [
        'x_EDA_mean','x_EDA_std','x_EDA_min','x_EDA_max','x_EDA_range','x_EDA_slope',
        'x_SCL_mean','x_SCL_std','x_SCL_slope','x_SCL_corr_time',
        'x_n_SCR','x_SCR_mean_amp','x_SCR_sum_amp','x_SCR_mean_dur','x_SCR_sum_dur','x_SCR_area',
    ]}
    return list({
        **_hrv_from_peaks([], 1, 'ecg'),
        **_hrv_from_peaks([], 1, 'bvp'),
        **{k.replace('x_', 'chest_'): v for k, v in _nan_eda.items()},
        **{k.replace('x_', 'wrist_'): v for k, v in _nan_eda.items()},
        **dict.fromkeys(_RESP_KEYS),
        **_temp_features(np.zeros(10), 'chest'),
        **_temp_features(np.zeros(10), 'wrist'),
        **dict.fromkeys(_EMG_KEYS),
        **_acc_features(np.zeros((10, 3), dtype=np.float32), 1, 'chest'),
        **_acc_features(np.zeros((10, 3), dtype=np.float32), 1, 'wrist'),
        **dict.fromkeys(_CROSS_KEYS),
    }.keys())

FEATURE_COLS = _dummy_feature_cols()
print(f"{len(FEATURE_COLS)} features defined")


# ── Subject loader + window finder ────────────────────────────────────────

def load_subject(sid):
    pkl_path = DATA_DIR / f"S{sid}" / f"S{sid}.pkl"
    with open(pkl_path, "rb") as f:
        raw = pickle.load(f, encoding="latin1")
    return {
        "chest":  {k: v.squeeze().astype(np.float32) for k, v in raw["signal"]["chest"].items()},
        "wrist":  {k: v.squeeze().astype(np.float32) for k, v in raw["signal"]["wrist"].items()},
        "labels": raw["label"].squeeze().astype(np.int8),
    }


def find_valid_starts(labels):
    n_steps    = (len(labels) - WIN_C) // STEP_C + 1
    change_pts = np.where(np.diff(labels) != 0)[0] + 1
    starts     = np.arange(n_steps, dtype=np.int32) * STEP_C
    ends       = starts + WIN_C
    cp_idx     = np.searchsorted(change_pts, starts + 1, side="left")
    has_cp     = cp_idx < len(change_pts)
    safe_idx   = np.minimum(cp_idx, len(change_pts) - 1)
    is_pure    = ~has_cp | (change_pts[safe_idx] >= ends)
    wlabel     = labels[starts]
    mask       = is_pure & np.isin(wlabel, list(KEEP_LABELS))
    return starts[mask], wlabel[mask]


# ── extract_features — uses pre-computed subject-level data ───────────────

def extract_features(chest, wrist, pc, cs):
    """
    pc : dict returned by precompute_subject()
    cs : window start index in 700 Hz chest samples
    """
    ce = cs + WIN_C

    # ── Index conversions ────────────────────────────────────────────
    cs_ecg  = round(cs * _ECG_DS_FS  / FS_CHEST)
    ce_ecg  = round(ce * _ECG_DS_FS  / FS_CHEST)
    cs_bvp  = round(cs * FS_BVP      / FS_CHEST)
    ce_bvp  = round(ce * FS_BVP      / FS_CHEST)
    cs_4    = round(cs * _EDA_FS     / FS_CHEST)
    ce_4    = round(ce * _EDA_FS     / FS_CHEST)
    cs_resp = round(cs * _RESP_FS_DS / FS_CHEST)
    ce_resp = round(ce * _RESP_FS_DS / FS_CHEST)
    cs_wacc = round(cs * FS_ACC_W    / FS_CHEST)
    ce_wacc = round(ce * FS_ACC_W    / FS_CHEST)
    cs_weda = round(cs * FS_EDA_W    / FS_CHEST)
    ce_weda = round(ce * FS_EDA_W    / FS_CHEST)
    cs_wtemp = round(cs * FS_TEMP_W  / FS_CHEST)
    ce_wtemp = round(ce * FS_TEMP_W  / FS_CHEST)

    # ── ECG HRV — slice pre-detected peaks to this window ────────────
    ecg_pk   = pc['ecg_peaks']
    ecg_win  = ecg_pk[(ecg_pk >= cs_ecg) & (ecg_pk < ce_ecg)] - cs_ecg
    ecg_f    = _hrv_from_peaks(ecg_win, _ECG_DS_FS, 'ecg')

    # ── BVP HRV — same pattern ────────────────────────────────────────
    bvp_pk   = pc['bvp_peaks']
    bvp_win  = bvp_pk[(bvp_pk >= cs_bvp) & (bvp_pk < ce_bvp)] - cs_bvp
    bvp_f    = _hrv_from_peaks(bvp_win, FS_BVP, 'bvp')

    # ── Chest EDA — slice pre-decomposed arrays ───────────────────────
    ceda_f = _eda_features_precomp(
        pc['eda_c_raw'][cs_4:ce_4]    if pc['eda_c_raw']    is not None else np.zeros(1),
        pc['eda_c_tonic'][cs_4:ce_4]  if pc['eda_c_tonic']  is not None else np.zeros(1),
        pc['eda_c_phasic'][cs_4:ce_4] if pc['eda_c_phasic'] is not None else np.zeros(1),
        pc['eda_c_df'].iloc[cs_4:ce_4] if pc['eda_c_df'] is not None else None,
        prefix='chest',
    )

    # ── Wrist EDA ─────────────────────────────────────────────────────
    weda_f = _eda_features_precomp(
        pc['eda_w_raw'][cs_weda:ce_weda]    if pc['eda_w_raw']    is not None else np.zeros(1),
        pc['eda_w_tonic'][cs_weda:ce_weda]  if pc['eda_w_tonic']  is not None else np.zeros(1),
        pc['eda_w_phasic'][cs_weda:ce_weda] if pc['eda_w_phasic'] is not None else np.zeros(1),
        pc['eda_w_df'].iloc[cs_weda:ce_weda] if pc['eda_w_df'] is not None else None,
        prefix='wrist',
    )

    # ── Respiration — slice pre-processed arrays ──────────────────────
    resp_f = _resp_features_precomp(
        raw_win      = chest['Resp'][cs:ce],
        resp_ds_win  = pc['resp_ds'][cs_resp:ce_resp]   if pc['resp_ds']   is not None else np.zeros(1),
        rsp_rate_win = pc['rsp_rate'][cs_resp:ce_resp]  if pc['rsp_rate']  is not None else None,
        rsp_phase_win= pc['rsp_phase'][cs_resp:ce_resp] if pc['rsp_phase'] is not None else None,
        rsp_peaks_full   = pc['rsp_peaks'],
        rsp_troughs_full = pc['rsp_troughs'],
        cs_r = cs_resp, ce_r = ce_resp,
    )

    # ── Temp, EMG, ACC — per-window (fast scipy ops) ──────────────────
    ctemp_f = _temp_features(chest['Temp'][cs:ce], 'chest')
    wtemp_f = _temp_features(wrist['TEMP'][cs_wtemp:ce_wtemp], 'wrist')
    emg_f   = _emg_features(chest['EMG'][cs:ce])
    cacc_f  = _acc_features(chest['ACC'][cs:ce], FS_CHEST, 'chest')
    wacc_f  = _acc_features(wrist['ACC'][cs_wacc:ce_wacc], FS_ACC_W, 'wrist')

    cross_f = _cross_features(ecg_f, resp_f)

    merged = {**ecg_f, **bvp_f, **ceda_f, **weda_f, **resp_f,
              **ctemp_f, **wtemp_f, **emg_f, **cacc_f, **wacc_f, **cross_f}
    return [merged[k] for k in FEATURE_COLS]

132 features defined


In [18]:
all_features = []
all_labels   = []
all_groups   = []

for sid in SUBJECT_IDS:
    print(f"S{sid}  pre-computing ...", end="  ", flush=True)

    data   = load_subject(sid)
    chest  = data["chest"]
    wrist  = data["wrist"]
    labels = data["labels"]

    # Run expensive nk calls once for the full subject signal
    pc = precompute_subject(chest, wrist)

    valid_starts, window_labels = find_valid_starts(labels)
    n_before = len(all_labels)

    for cs, lbl in zip(valid_starts, window_labels):
        features = extract_features(chest, wrist, pc, int(cs))
        all_features.append(features)
        all_labels.append(1 if lbl == 2 else 0)   # stress=1, baseline/amusement=0
        all_groups.append(sid)

    n_added  = len(all_labels) - n_before
    n_stress = sum(all_labels[n_before:])
    print(f"{n_added} windows  (stress={n_stress}, non-stress={n_added - n_stress})")

    del data, chest, wrist, labels, pc
    gc.collect()

X      = np.array(all_features, dtype=np.float32)
y      = np.array(all_labels,   dtype=np.int8)
groups = np.array(all_groups,   dtype=np.int32)

print(f"\nX      : {X.shape}  ({X.nbytes / 1e6:.1f} MB)")
print(f"y      : {y.shape}  —  stress={y.sum()}, non-stress={(y == 0).sum()}")
print(f"groups : {np.unique(groups).tolist()}")

S2  pre-computing ...  

/var/folders/46/3phyn9592wj1fgj0kywgvld00000gq/T/ipykernel_10799/194572340.py:405: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  raw = pickle.load(f, encoding="latin1")
/Users/dylannguyen/Desktop/CS 372/finalproject/wesad/.venv/lib/python3.12/site-packages/neurokit2/eda/eda_clean.py:102: NeuroKitWarning: EDA signal is sampled at very low frequency. Skipping filtering.
  warn(
/Users/dylannguyen/Desktop/CS 372/finalproject/wesad/.venv/lib/python3.12/site-packages/neurokit2/eda/eda_clean.py:102: NeuroKitWarning: EDA signal is sampled at very low frequency. Skipping filtering.
  warn(


7764 windows  (stress=2220, non-stress=5544)
S3  pre-computing ...  

/var/folders/46/3phyn9592wj1fgj0kywgvld00000gq/T/ipykernel_10799/194572340.py:405: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  raw = pickle.load(f, encoding="latin1")
/Users/dylannguyen/Desktop/CS 372/finalproject/wesad/.venv/lib/python3.12/site-packages/neurokit2/eda/eda_clean.py:102: NeuroKitWarning: EDA signal is sampled at very low frequency. Skipping filtering.
  warn(
/Users/dylannguyen/Desktop/CS 372/finalproject/wesad/.venv/lib/python3.12/site-packages/neurokit2/eda/eda_clean.py:102: NeuroKitWarning: EDA signal is sampled at very low frequency. Skipping filtering.
  warn(


7900 windows  (stress=2320, non-stress=5580)
S4  pre-computing ...  

...

8212 windows  (stress=2504, non-stress=5708)
S16  pre-computing ...  

/var/folders/46/3phyn9592wj1fgj0kywgvld00000gq/T/ipykernel_10799/194572340.py:405: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  raw = pickle.load(f, encoding="latin1")
/Users/dylannguyen/Desktop/CS 372/finalproject/wesad/.venv/lib/python3.12/site-packages/neurokit2/eda/eda_clean.py:102: NeuroKitWarning: EDA signal is sampled at very low frequency. Skipping filtering.
  warn(
/Users/dylannguyen/Desktop/CS 372/finalproject/wesad/.venv/lib/python3.12/site-packages/neurokit2/eda/eda_clean.py:102: NeuroKitWarning: EDA signal is sampled at very low frequency. Skipping filtering.
  warn(


8164 windows  (stress=2452, non-stress=5712)
S17  pre-computing ...  

/var/folders/46/3phyn9592wj1fgj0kywgvld00000gq/T/ipykernel_10799/194572340.py:405: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  raw = pickle.load(f, encoding="latin1")
/Users/dylannguyen/Desktop/CS 372/finalproject/wesad/.venv/lib/python3.12/site-packages/neurokit2/eda/eda_clean.py:102: NeuroKitWarning: EDA signal is sampled at very low frequency. Skipping filtering.
  warn(
/Users/dylannguyen/Desktop/CS 372/finalproject/wesad/.venv/lib/python3.12/site-packages/neurokit2/eda/eda_clean.py:102: NeuroKitWarning: EDA signal is sampled at very low frequency. Skipping filtering.
  warn(


8384 windows  (stress=2652, non-stress=5732)

X      : (121808, 132)  (64.3 MB)
y      : (121808,)  —  stress=36264, non-stress=85544
groups : [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]


### Sanity checks

In [21]:
import pandas as pd

assert X.shape[1] == len(FEATURE_COLS), "Feature count mismatch"
assert len(X) == len(y) == len(groups),  "Array length mismatch"
assert not np.any(np.isinf(X)),          "Infs in X"

nan_counts = np.isnan(X).sum(axis=0)
n_nan_feat = (nan_counts > 0).sum()
n_nan_win  = np.isnan(X).any(axis=1).sum()
print(f"Features with any NaN : {n_nan_feat} / {len(FEATURE_COLS)}")
print(f"Windows   with any NaN : {n_nan_win} / {len(X)}  ({100*n_nan_win/len(X):.1f}%)")
if n_nan_feat:
    top = pd.Series(nan_counts, index=FEATURE_COLS).nlargest(10)
    print("\nTop NaN features:")
    print(top.to_string())

print(f"\nAll good — {len(FEATURE_COLS)} features, {len(X)} windows\n")

df = pd.DataFrame(X, columns=FEATURE_COLS)
df["y"]       = y
df["subject"] = groups

print("Windows per subject:")
print(df.groupby("subject")["y"].value_counts().unstack(fill_value=0)
        .rename(columns={0: "non-stress", 1: "stress"}))

Features with any NaN : 32 / 132
Windows   with any NaN : 121808 / 121808  (100.0%)

Top NaN features:
chest_SCR_mean_dur    121808
chest_SCR_sum_dur     121808
chest_SCR_area        121808
wrist_SCR_mean_dur    121808
wrist_SCR_sum_dur     121808
wrist_SCR_area        121808
resp_mean_inhale      121808
resp_std_inhale       121808
resp_mean_exhale      121808
resp_std_exhale       121808

All good — 132 features, 121808 windows

Windows per subject:
y        non-stress  stress
subject                    
2              5544    2220
3              5580    2320
4              5640    2300
5              5808    2340
6              5728    2360
7              5752    2320
8              5676    2440
9              5728    2340
10             5728    2660
11             5712    2480
13             5768    2416
14             5728    2460
15             5708    2504
16             5712    2452
17             5732    2652


In [22]:
# ── Drop features that are ≥ 99 % NaN ────────────────────────────────────
# 100 % NaN = structurally broken (column name mismatch / empty peak arrays).
# Partial-NaN features are kept — EBM handles them natively via surrogate splits.

nan_pct  = np.isnan(X).mean(axis=0) * 100
dead     = [c for c, p in zip(FEATURE_COLS, nan_pct) if p >= 99]
keep_idx = [i for i, p in enumerate(nan_pct) if p < 99]
keep     = [FEATURE_COLS[i] for i in keep_idx]

print(f"Dropping {len(dead)} always-NaN features:")
for c in dead:
    print(f"  {c}")

X            = X[:, keep_idx]
FEATURE_COLS = keep

nan_pct_k  = np.isnan(X).mean(axis=0) * 100
n_partial  = (nan_pct_k > 0).sum()
n_win_nan  = np.isnan(X).any(axis=1).sum()
print(f"\nAfter drop : {len(FEATURE_COLS)} features")
print(f"Partial-NaN: {n_partial} features  ({n_win_nan}/{len(X)} windows have ≥1 NaN — EBM handles these)")

# ── Scale summary ─────────────────────────────────────────────────────────
# EBM is tree-based → scale-invariant. This is purely informational.
scale_df = pd.DataFrame({
    'nan_%': np.round(nan_pct_k, 1),
    'mean':  np.round(np.nanmean(X, axis=0), 4),
    'std':   np.round(np.nanstd(X,  axis=0), 4),
    'min':   np.round(np.nanmin(X,  axis=0), 4),
    'max':   np.round(np.nanmax(X,  axis=0), 4),
}, index=FEATURE_COLS)

print("\nScale summary — top 20 by std:")
print(scale_df.sort_values('std', ascending=False).head(20).to_string())

Dropping 13 always-NaN features:
  chest_SCR_mean_dur
  chest_SCR_sum_dur
  chest_SCR_area
  wrist_SCR_mean_dur
  wrist_SCR_sum_dur
  wrist_SCR_area
  resp_mean_inhale
  resp_std_inhale
  resp_mean_exhale
  resp_std_exhale
  resp_IE_ratio
  resp_inspiration_vol
  resp_sum_dur

After drop : 119 features
Partial-NaN: 19 features  (108424/121808 windows have ≥1 NaN — EBM handles these)

Scale summary — top 20 by std:
                      nan_%          mean           std          min           max
bvp_TP                  0.5  4.771136e+06  1.552336e+07  7209.312988  3.712471e+08
bvp_LF                  0.5  3.187598e+06  1.100031e+07     0.000000  3.271555e+08
bvp_HF                  0.5  9.257007e+05  3.228711e+06  1745.546753  1.004118e+08
ecg_TP                  0.0  5.095235e+04  4.709828e+04   641.122925  4.613338e+05
ecg_LF                  0.0  3.307395e+04  3.105081e+04   148.976593  3.428689e+05
ecg_HF                  0.0  1.232711e+04  1.576097e+04    59.474998  1.284981e+05
c

In [23]:
import json

PROCESSED_DIR = DATA_DIR / "processed_engineered_dropnan"
PROCESSED_DIR.mkdir(exist_ok=True)

np.save(PROCESSED_DIR / "X.npy",      X)
np.save(PROCESSED_DIR / "y.npy",      y)
np.save(PROCESSED_DIR / "groups.npy", groups)

with open(PROCESSED_DIR / "feature_cols.json", "w") as f:
    json.dump(FEATURE_COLS, f)

print(f"Saved to {PROCESSED_DIR}/")
print(f"  X.npy           {X.shape}  {X.nbytes / 1e6:.1f} MB")
print(f"  y.npy           {y.shape}")
print(f"  groups.npy      {groups.shape}")
print(f"  feature_cols.json  {len(FEATURE_COLS)} features")

Saved to ../data/WESAD/processed_engineered_dropnan/
  X.npy           (121808, 119)  58.0 MB
  y.npy           (121808,)
  groups.npy      (121808,)
  feature_cols.json  119 features
